<a href="https://colab.research.google.com/github/jimmyGit538/coin-market-cap-project/blob/production/Jessica/CMC_Category_Focused_Historical_30D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#remove this for couldrun

from google.colab import auth
auth.authenticate_user()
print("Authenticated")


Authenticated


In [ ]:

# Install and import libraries


!pip install --quiet requests pandas google-cloud-bigquery db-dtypes

import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from getpass import getpass

from google.cloud import bigquery


# Configuration section


# GCP project
PROJECT_ID = "coinmarketcapproject"

# Dataset where new tables will live
DATASET_ID = "crypto_raw"

# Input table (already filtered to 29 categories)
TABLE_CATEGORY_TOP20_COINS_29 = "category_top20_coins_29_raw"

# Output table
TABLE_QUOTES_HIST_30D_29 = "quotes_historical_30d_29_raw"

BASE_URL_V3 = "https://pro-api.coinmarketcap.com"
HIST_URL = f"{BASE_URL_V3}/v3/cryptocurrency/quotes/historical"

# prompts securely.
CMC_API_KEY = getpass("Enter your CoinMarketCap API key: ").strip()

# Common request headers for CMC
HEADERS = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": CMC_API_KEY
}

# BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print("Reading:", f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CATEGORY_TOP20_COINS_29}")
print("Writing:", f"{PROJECT_ID}.{DATASET_ID}.{TABLE_QUOTES_HIST_30D_29}")


Enter your CoinMarketCap API key: ··········
Reading: coinmarketcapproject.crypto_raw.category_top20_coins_29_raw
Writing: coinmarketcapproject.crypto_raw.quotes_historical_30d_29_raw


In [ ]:
#Helper functions (chunking, iso format, BigQuery write)

def iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

def chunk_list(items, size=50):
    for i in range(0, len(items), size):
        yield items[i:i+size]

def write_df_to_bigquery(df, table_name, write_disposition="WRITE_APPEND"):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True
    )
    job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"Loaded {len(df)} rows into {table_id}")


In [ ]:
#Pull category + symbol mapping from BigQuery

sql = f"""
WITH ranked AS (
  SELECT
    CAST(category_id AS STRING) AS category_id,
    SAFE_CAST(coin_id AS INT64) AS coin_id,
    coin_symbol,
    coin_name,
    coin_cmc_rank,
    ROW_NUMBER() OVER (
      PARTITION BY category_id
      ORDER BY coin_cmc_rank ASC
    ) AS rn
  FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CATEGORY_TOP20_COINS_29}`
  WHERE coin_id IS NOT NULL
)
SELECT *
FROM ranked
WHERE rn <= 20
"""

df_focus = bq_client.query(sql).to_dataframe()

print("Rows (category x top20):", len(df_focus))
print("Distinct categories:", df_focus["category_id"].nunique())
print("Distinct coin_ids:", df_focus["coin_id"].nunique())

df_focus.head()




Rows (category x top20): 576
Distinct categories: 29
Distinct coin_ids: 483


,category_id,coin_id,coin_symbol,coin_name,coin_cmc_rank,rn
0,5fb62883c9ddcc213ed13308,32554,wIOTA,wIOTA,NaN,1
1,5fb62883c9ddcc213ed13308,37574,TETH,Treehouse ETH,NaN,2
2,5fb62883c9ddcc213ed13308,5830,NXM,NXM,NaN,3
3,5fb62883c9ddcc213ed13308,23177,SFRXETH,Frax Staked Ether,NaN,4
4,5fb62883c9ddcc213ed13308,6432,OMI,ECOMI,NaN,5


In [ ]:
coin_ids = sorted(df_focus["coin_id"].dropna().unique().tolist())
print("Unique coin_ids to pull history for:", len(coin_ids))
coin_ids[:20]



Unique coin_ids to pull history for: 483


[1,
 2,
 74,
 131,
 291,
 328,
 512,
 623,
 626,
 693,
 825,
 954,
 1027,
 1050,
 1168,
 1321,
 1340,
 1405,
 1414,
 1423]

In [ ]:
#historical block


def fetch_quotes_historical_by_id(coin_id_batch, interval="daily", convert="USD"):
    from datetime import datetime, timezone
    #edit this dates to bring each month individually.
    time_start = datetime(2025, 1, 1, tzinfo=timezone.utc)
    time_end   = datetime(2025, 1, 20, tzinfo=timezone.utc)


    params = {
    "id": ",".join(str(x) for x in coin_id_batch),
    "time_start": iso_z(time_start),
    "time_end": iso_z(time_end),
    "interval": interval,
    "convert": convert,
    "aux": "price,volume,market_cap,circulating_supply,total_supply,quote_timestamp,search_interval",
    "skip_invalid": "true"
}


    max_retries = 5
    for attempt in range(1, max_retries + 1):
        r = requests.get(HIST_URL, headers=HEADERS, params=params, timeout=60)

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429:
            print(f"429 rate limit. Waiting 60s... (attempt {attempt}/{max_retries})")
            time.sleep(60)
            continue

        print("Request failed:", r.status_code, r.text[:500])
        r.raise_for_status()

    raise RuntimeError("Exceeded max retries due to rate limiting (429).")



In [ ]:
def flatten_historical_payload_id(payload, convert="USD"):
    rows = []
    data = payload.get("data", {}) if isinstance(payload, dict) else {}

    for coin_id_str, obj in data.items():
        if not isinstance(obj, dict):
            continue

        cid = obj.get("id") or coin_id_str
        sym = obj.get("symbol")

        quotes = obj.get("quotes", []) or []
        for q in quotes:
            usd = (q.get("quote") or {}).get(convert, {}) or {}

            rows.append({
                "coin_id": int(cid),
                "coin_symbol": sym,
                "quote_timestamp": q.get("timestamp"),
                "price": usd.get("price"),
                "volume_24h": usd.get("volume_24h"),
                "market_cap": usd.get("market_cap"),
                "circulating_supply": usd.get("circulating_supply"),
                "total_supply": usd.get("total_supply"),
                "search_interval": q.get("search_interval"),
            })

    return rows



In [ ]:

import time

all_rows = []
total_batches = (len(coin_ids) + 24) // 25
print("Total batches:", total_batches)

for i, batch in enumerate(chunk_list(coin_ids, size=25), start=1):
    print(f"Batch {i}/{total_batches} starting (coin_ids {len(batch)})")

    payload = fetch_quotes_historical_by_id(batch, interval="daily", convert="USD")
    rows = flatten_historical_payload_id(payload, convert="USD")
    all_rows.extend(rows)

    print(f"Batch {i} done. Added {len(rows)} rows. Total rows: {len(all_rows)}")
    time.sleep(12)  # throttle for hobbyist

df_hist = pd.DataFrame(all_rows)
df_hist["quote_timestamp"] = pd.to_datetime(df_hist["quote_timestamp"], errors="coerce")
df_hist["imported_at_utc"] = datetime.now(timezone.utc)

print("Done. Rows:", len(df_hist))
df_hist.head()



Total batches: 20
Batch 1/20 starting (coin_ids 25)
Batch 1 done. Added 304 rows. Total rows: 304
Batch 2/20 starting (coin_ids 25)
Batch 2 done. Added 279 rows. Total rows: 583
Batch 3/20 starting (coin_ids 25)
Batch 3 done. Added 228 rows. Total rows: 811
Batch 4/20 starting (coin_ids 25)
Batch 4 done. Added 259 rows. Total rows: 1070
Batch 5/20 starting (coin_ids 25)
Batch 5 done. Added 285 rows. Total rows: 1355
Batch 6/20 starting (coin_ids 25)
Batch 6 done. Added 304 rows. Total rows: 1659
Batch 7/20 starting (coin_ids 25)
Batch 7 done. Added 361 rows. Total rows: 2020
Batch 8/20 starting (coin_ids 25)
Batch 8 done. Added 190 rows. Total rows: 2210
Batch 9/20 starting (coin_ids 25)
Batch 9 done. Added 304 rows. Total rows: 2514
Batch 10/20 starting (coin_ids 25)
Batch 10 done. Added 215 rows. Total rows: 2729
Batch 11/20 starting (coin_ids 25)
Batch 11 done. Added 285 rows. Total rows: 3014
Batch 12/20 starting (coin_ids 25)
Batch 12 done. Added 285 rows. Total rows: 3299
Batch 1

,coin_id,coin_symbol,quote_timestamp,price,volume_24h,market_cap,circulating_supply,total_supply,search_interval,imported_at_utc
0,1,BTC,2025-01-02 00:00:00+00:00,94390.274024,2.452391e+10,1.869335e+12,19804318.0,19804318.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00
1,1,BTC,2025-01-03 00:00:00+00:00,96890.004327,4.600740e+10,1.918894e+12,19804871.0,19804871.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00
2,1,BTC,2025-01-04 00:00:00+00:00,98167.573838,3.561167e+10,1.944239e+12,19805312.0,19805312.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00
3,1,BTC,2025-01-05 00:00:00+00:00,98211.267775,2.234123e+10,1.945150e+12,19805771.0,19805771.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00
4,1,BTC,2025-01-06 00:00:00+00:00,98336.037415,2.053634e+10,1.947670e+12,19806271.0,19806271.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00


In [ ]:
id_to_symbol = df_focus.dropna(subset=["coin_id", "coin_symbol"]).set_index("coin_id")["coin_symbol"].to_dict()
df_hist["coin_symbol"] = df_hist["coin_symbol"].fillna(df_hist["coin_id"].map(id_to_symbol))


In [ ]:
id_to_cats = (
    df_focus.groupby("coin_id")["category_id"]
    .apply(lambda s: sorted(set(s.tolist())))
    .to_dict()
)

expanded = []
for _, row in df_hist.iterrows():
    cats = id_to_cats.get(row["coin_id"], [])
    for cid in cats:
        out = row.to_dict()
        out["category_id"] = str(cid)
        expanded.append(out)

df_hist_by_category = pd.DataFrame(expanded)
print("Rows after attaching categories:", len(df_hist_by_category))
df_hist_by_category.head()


Rows after attaching categories: 6632


,coin_id,coin_symbol,quote_timestamp,price,volume_24h,market_cap,circulating_supply,total_supply,search_interval,imported_at_utc,category_id
0,1,BTC,2025-01-02 00:00:00+00:00,94390.274024,2.452391e+10,1.869335e+12,19804318.0,19804318.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00,6433de7df79a2653906cd680
1,1,BTC,2025-01-03 00:00:00+00:00,96890.004327,4.600740e+10,1.918894e+12,19804871.0,19804871.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00,6433de7df79a2653906cd680
2,1,BTC,2025-01-04 00:00:00+00:00,98167.573838,3.561167e+10,1.944239e+12,19805312.0,19805312.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00,6433de7df79a2653906cd680
3,1,BTC,2025-01-05 00:00:00+00:00,98211.267775,2.234123e+10,1.945150e+12,19805771.0,19805771.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00,6433de7df79a2653906cd680
4,1,BTC,2025-01-06 00:00:00+00:00,98336.037415,2.053634e+10,1.947670e+12,19806271.0,19806271.0,2025-01-02T00:00:00.000Z,2025-12-23 22:43:05.708985+00:00,6433de7df79a2653906cd680


In [ ]:
write_df_to_bigquery(
    df_hist_by_category,
    TABLE_QUOTES_HIST_30D_29,
    write_disposition="WRITE_APPEND"
)



Loaded 6632 rows into coinmarketcapproject.crypto_raw.quotes_historical_30d_29_raw


In [ ]:
test_payload = fetch_quotes_historical_by_id(coin_ids[:1], interval="daily", convert="USD")
rows = flatten_historical_payload_id(test_payload, convert="USD")
df_test = pd.DataFrame(rows)

df_test[["coin_id","quote_timestamp","price","market_cap","volume_24h"]].head()


,coin_id,quote_timestamp,price,market_cap,volume_24h
0,1,2025-01-02T00:00:00.000Z,94390.274024,1.869335e+12,2.452391e+10
1,1,2025-01-03T00:00:00.000Z,96890.004327,1.918894e+12,4.600740e+10
2,1,2025-01-04T00:00:00.000Z,98167.573838,1.944239e+12,3.561167e+10
3,1,2025-01-05T00:00:00.000Z,98211.267775,1.945150e+12,2.234123e+10
4,1,2025-01-06T00:00:00.000Z,98336.037415,1.947670e+12,2.053634e+10
